In [ ]:
import pandas as pd
import urllib.request
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

base_dir = '/Users/krishsati13/Desktop/GSoC_ArtExtract/data/nga'
image_dir = os.path.join(base_dir, 'images')
os.makedirs(image_dir, exist_ok=True)

print(f"📁 Saving everything permanently to: {image_dir}")

objects_path = os.path.join(base_dir, 'objects.csv')
images_path = os.path.join(base_dir, 'published_images.csv')

df_objects = pd.read_csv(objects_path, low_memory=False)
df_images = pd.read_csv(images_path, low_memory=False)

df_merged = df_objects.merge(df_images, left_on='objectid', right_on='depictstmsobjectid')

df_paintings = df_merged[df_merged['medium'].str.contains('oil|canvas|panel|fresco|watercolor|paint', na=False, case=False)].reset_index(drop=True)


df_paintings.to_csv(os.path.join(base_dir, 'nga_full_dataset_metadata.csv'), index=False)
print(f"🎯 Target Locked: {len(df_paintings)} paintings.")

print("⏳ Initiating Mass Download (Leave your Mac plugged in and awake!)...")

error_count = 0 

def download_image(row):
    global error_count
    if pd.isna(row['iiifthumburl']): return
    
   
    image_url = str(row['iiifthumburl']).replace("!200,200", "!500,500") 
    image_name = f"{row['objectid']}.jpg"
    image_path = os.path.join(image_dir, image_name)
    
   
    if not os.path.exists(image_path):
        try:
            req = urllib.request.Request(image_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response, open(image_path, 'wb') as out_file:
                out_file.write(response.read())
        except Exception as e:
            error_count += 1

with ThreadPoolExecutor(max_workers=15) as executor:
    list(tqdm(executor.map(download_image, df_paintings.iloc), total=len(df_paintings)))


actual_downloaded = len([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
print(f"\n✅ Download Process Complete!")
print(f"🖼️ ACTUALLY Downloaded: {actual_downloaded} images")
if error_count > 0:
    print(f"⚠️ Museum server skipped {error_count} broken links.")

📁 Saving everything permanently to: /Users/krishsati13/Desktop/GSoC_ArtExtract/data/nga/images
🎯 Target Locked: 23449 paintings.
⏳ Initiating Mass Download (Leave your Mac plugged in and awake!)...


  0%|          | 0/23449 [00:00<?, ?it/s]


✅ Download Process Complete!
🖼️ ACTUALLY Downloaded: 22968 images
⚠️ Museum server skipped 144 broken links.


In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
from tqdm.notebook import tqdm
import ssl


ssl._create_default_https_context = ssl._create_unverified_context


save_dir = '/Users/krishsati13/Desktop/GSoC_ArtExtract/data/nga'
image_dir = '/Users/krishsati13/Desktop/GSoC_ArtExtract/data/nga/images'


print("🧹 Sweeping for corrupted museum files...")
all_files = [f for f in os.listdir(image_dir) if f.lower().endswith('.jpg')]
valid_files = []
corrupted_count = 0

for f in tqdm(all_files, desc="Checking image integrity"):
    img_path = os.path.join(image_dir, f)
    try:
        # .verify() quickly checks the file header without loading the heavy pixels
        with Image.open(img_path) as img:
            img.verify()
        valid_files.append(f)
    except Exception:
        corrupted_count += 1
        os.remove(img_path) # Delete the fake/corrupted image

print(f"🛡️ Integrity Check Complete: Kept {len(valid_files)} perfect images. Trashed {corrupted_count} corrupted files.")

if len(valid_files) == 0:
    raise FileNotFoundError("No valid images left!")


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"⚡ Using compute device: {device}")

print("🧠 Initializing ResNet50 Feature Extractor...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Identity() 
model = model.to(device)
model.eval() 

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class NGADataset(Dataset):
    def __init__(self, image_dir, image_files, transform=None):
        self.image_dir = image_dir
        self.image_files = image_files
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        object_id = img_name.split('.')[0] 
        return image, object_id

nga_dataset = NGADataset(image_dir=image_dir, image_files=valid_files, transform=transform)


dataloader = DataLoader(nga_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"🚀 Starting mathematical extraction on {len(nga_dataset)} paintings...")

all_vectors = []
all_ids = []


with torch.no_grad(): 
    for images, object_ids in tqdm(dataloader, desc="Extracting Features"):
        images = images.to(device)
        features = model(images)
        all_vectors.append(features.cpu().numpy())
        all_ids.extend(object_ids)

vector_matrix = np.vstack(all_vectors)
np.save(os.path.join(save_dir, 'nga_vectors.npy'), vector_matrix)
np.save(os.path.join(save_dir, 'nga_vector_ids.npy'), np.array(all_ids))

print(f"✅ Extraction Complete! Saved a massive Matrix of shape: {vector_matrix.shape}")

🧹 Sweeping for corrupted museum files...


Checking image integrity:   0%|          | 0/22842 [00:00<?, ?it/s]

🛡️ Integrity Check Complete: Kept 22842 perfect images. Trashed 0 corrupted files.
⚡ Using compute device: mps
🧠 Initializing ResNet50 Feature Extractor...
🚀 Starting mathematical extraction on 22842 paintings...


Extracting Features:   0%|          | 0/357 [00:00<?, ?it/s]

✅ Extraction Complete! Saved a massive Matrix of shape: (22842, 2048)
